<a href="https://colab.research.google.com/github/D3zNt/Team3AmazonProject/blob/feature%2FYiranQi/try_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# YOLOv8 Training with COCO8 Dataset
# Save this as: notebooks/yolo_training.ipynb
!pip install ultralytics
import os
from pathlib import Path
import yaml
from ultralytics import YOLO
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

import torch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 102.7 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12

In [2]:
import zipfile, tarfile, pathlib, os, binascii
from google.colab import drive
drive.mount('/content/drive')# mount drive
def smart_extract(path, out_dir):
    path = pathlib.Path(path)
    out_dir = pathlib.Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    if not path.exists() or path.stat().st_size < 64:
        raise RuntimeError(f"too small or do not exist: {path} (size={path.stat().st_size if path.exists() else 'NA'})")

    if zipfile.is_zipfile(path):
        with zipfile.ZipFile(path, 'r') as zf:
            zf.extractall(out_dir)
        return f"zip accomplished → {out_dir}"

    if tarfile.is_tarfile(path):
        with tarfile.open(path, 'r:*') as tf:
            tf.extractall(out_dir)
        return f"tar accomplished → {out_dir}"

    with open(path,'rb') as f:
        head = f.read(32)
    raise RuntimeError(f"ERROR magic={binascii.hexlify(head)}")

print(smart_extract("/content/drive/MyDrive/mixed.zip", "/content/mixed"))


Mounted at /content/drive
zip accomplished → /content/mixed


In [10]:
from pathlib import Path
import yaml

BASE = Path("/content/mixed/mixed")          # 数据根目录
SPL  = BASE / "splits"
OLD  = "D:/PBL_Amazon/dataset"               # 旧 Windows 前缀
NEW  = str(BASE)                             # 新前缀（POSIX 绝对路径）

def fix_list(p):
    p = Path(p)
    if not p.exists():
        print("skip (no file):", p); return
    lines = [l.strip() for l in p.read_text(encoding="utf-8").splitlines() if l.strip()]
    lines = [l.replace("\\", "/").replace(OLD, NEW) for l in lines]
    p.write_text("\n".join(lines), encoding="utf-8")
    print("fixed list:", p)

for name in ["train_oversample_1to1.txt","train_oversample_1to2.txt",
             "train.txt","val.txt","test.txt","val_day.txt","val_night.txt"]:
    fix_list(SPL / name)

def rewrite_yaml(yaml_path: Path, train_list_name: str):
    with open(yaml_path, "r", encoding="utf-8") as f:
        data = yaml.safe_load(f)
    data["path"]  = str(BASE).replace("\\","/")  # 数据根
    data["train"] = str((SPL / train_list_name).resolve()).replace("\\","/")
    data["val"]   = str((SPL / "val.txt").resolve()).replace("\\","/")
    data["test"]  = str((SPL / "test.txt").resolve()).replace("\\","/")
    with open(yaml_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(data, f, allow_unicode=True, sort_keys=False)
    print("fixed yaml:", yaml_path)

rewrite_yaml(BASE / "daynight_train_1to1.yaml", "train_oversample_1to1.txt")
rewrite_yaml(BASE / "daynight_train_1to2.yaml", "train_oversample_1to2.txt")


fixed list: /content/mixed/mixed/splits/train_oversample_1to1.txt
fixed list: /content/mixed/mixed/splits/train_oversample_1to2.txt
fixed list: /content/mixed/mixed/splits/train.txt
fixed list: /content/mixed/mixed/splits/val.txt
fixed list: /content/mixed/mixed/splits/test.txt
fixed list: /content/mixed/mixed/splits/val_day.txt
fixed list: /content/mixed/mixed/splits/val_night.txt
fixed yaml: /content/mixed/mixed/daynight_train_1to1.yaml
fixed yaml: /content/mixed/mixed/daynight_train_1to2.yaml


In [11]:
!rm -f /content/mixed/mixed/labels/*.cache


In [12]:
from pathlib import Path

DATA_BASE = "/content/mixed/mixed"
round_ndec = 6                      # 取6位小数去重，避免极微小浮点差

def clean_label_file(p: Path):
    raw = p.read_text(encoding="utf-8").splitlines()
    uniq = []
    seen = set()
    removed_dup = 0
    removed_bad = 0

    def valid_xywh(x, y, w, h):
        return (0 <= x <= 1) and (0 <= y <= 1) and (0 < w <= 1) and (0 < h <= 1)

    for line in raw:
        s = line.strip()
        if not s:
            continue
        parts = s.split()
        if len(parts) != 5:
            removed_bad += 1
            continue
        cls = parts[0]
        try:
            x, y, w, h = map(float, parts[1:])
        except:
            removed_bad += 1
            continue
        if not valid_xywh(x, y, w, h):
            removed_bad += 1
            continue
        key = (cls,
               round(x, round_ndec),
               round(y, round_ndec),
               round(w, round_ndec),
               round(h, round_ndec))
        if key in seen:
            removed_dup += 1
            continue
        seen.add(key)
        uniq.append(f"{cls} {key[1]} {key[2]} {key[3]} {key[4]}")

    if removed_dup or removed_bad:
        p.write_text("\n".join(uniq) + ("\n" if uniq else ""), encoding="utf-8")
    return removed_dup, removed_bad

tot_dup = tot_bad = 0
for split in ["train", "val", "test"]:
    lbl_dir = Path(DATA_BASE)/"labels"/split
    if not lbl_dir.exists():
        continue
    for f in lbl_dir.glob("*.txt"):
        d, b = clean_label_file(f)
        tot_dup += d; tot_bad += b

print(f"Removed duplicates: {tot_dup}, removed invalid: {tot_bad}")


Removed duplicates: 0, removed invalid: 0


## Please run all the codes before this cell

In [ ]:
from ultralytics import YOLO
import torch, yaml, json, os
from datetime import datetime
import numpy as np, torch
from ultralytics.utils import LOGGER
import albumentations as A
# ====== paths ======
DATA_BASE = r"/content/mixed/mixed"
DATA_TRAIN = fr"{DATA_BASE}/daynight_train_1to1.yaml"   # 用 1:2 过采样清单训练；要 1:1 就换成 daynight_train_1to1.yaml
DATA_BASE_YAML = fr"{DATA_BASE}/daynight.yaml"
VAL_DAY_LIST   = fr"{DATA_BASE}/splits/val_day.txt"
VAL_NIGHT_LIST = fr"{DATA_BASE}/splits/val_night.txt"
old_prefix = r"D:/PBL_Amazon/dataset"
new_prefix = DATA_BASE


# ====== config ======
now = datetime.now().strftime("%Y%m%d_%H%M%S")
cfg = dict(
  model="yolov8s.pt",
  data=DATA_BASE_YAML,
  epochs=50, imgsz=896, batch=16, workers=2,
  device = "cuda" if torch.cuda.is_available() else "cpu", amp=True, pretrained=True,
  optimizer="SGD", lr0=0.005, lrf=0.01, cos_lr=True, momentum=0.937, weight_decay=5e-4,
  warmup_epochs=3.0, warmup_momentum=0.8, warmup_bias_lr=0.1,
  mosaic=0.7, mixup=0.1, copy_paste=0.1, erasing=0.15,
  hsv_h=0.015, hsv_s=0.5, hsv_v=0.2,
  translate=0.1, scale=0.5, fliplr=0.5, degrees=0.0, shear=0.0, perspective=0.0,
  close_mosaic=10, rect=False,
  project="runs_daynight", name="dn_baseline", exist_ok=True,
  patience=30
)

# ==================================================

APPLY_PROB = 0.30        # 由 0.5 降到 0.30
MAX_SCALE  = 0.20        # 由 0.30 降到 0.20
START_EP   = 5           # 5~30 之间启用
STOP_EP    = 30

def build_night_aug(scale=0.2):
    scale = float(np.clip(scale, 0.0, 0.5))
    ps = {
        "gamma": 0.5 * scale,
        "noise": 0.4 * scale,
        "motion": 0.25 * scale,
        "shadow": 0.20 * scale,
        "fog": 0.10 * scale,
    }
    return A.Compose([
        A.RandomGamma(gamma_limit=(80, 120), p=ps["gamma"]),
        A.ISONoise(color_shift=(0.01, 0.04), intensity=(0.08, 0.3), p=ps["noise"]),
        A.MotionBlur(blur_limit=(3, 5), p=ps["motion"]),
        A.RandomShadow(num_shadows_limit=(1, 2), shadow_dimension=5, p=ps["shadow"]),
        A.RandomFog(fog_coef_range=(0.08, 0.16), alpha_coef=0.06, p=ps["fog"]),
    ])

AUG_STATE = {"scale": 0.0, "aug": build_night_aug(0.0)}

def is_bright(img_tensor, thr=0.60):
    t = img_tensor.float()
    if t.max() > 1.5: t = t / 255.0
    r,g,b = t[0], t[1], t[2]
    lum = 0.2126*r + 0.7152*g + 0.0722*b
    return float(lum.mean().item()) > thr

def on_train_epoch_start(trainer):
    e = int(getattr(trainer, "epoch", 0)) + 1
    if e < START_EP:
        AUG_STATE["scale"] = 0.0
    elif e <= STOP_EP:
        # 线性爬坡到 MAX_SCALE
        s = MAX_SCALE * (e-START_EP) / max(1, (STOP_EP-START_EP))
        AUG_STATE["scale"] = s
    else:
        AUG_STATE["scale"] = 0.0
    AUG_STATE["aug"] = build_night_aug(AUG_STATE["scale"])
    LOGGER.info(f"[NightAug] epoch={e} scale={AUG_STATE['scale']:.2f}")

def on_train_batch_start(trainer):
    if AUG_STATE["scale"] <= 1e-6:
        return
    batch = getattr(trainer, "batch", None)
    if not isinstance(batch, dict) or "img" not in batch:
        return
    imgs = batch["img"].float()
    scaled_to_01 = False
    if imgs.max() > 1.5: imgs /= 255.0; scaled_to_01 = True

    applied = 0
    for i in range(imgs.shape[0]):
        if np.random.rand() > APPLY_PROB:
            continue
        im = imgs[i]
        if not is_bright(im, thr=0.60):
            continue
        np_img = (im.detach().cpu().permute(1,2,0).numpy()*255).clip(0,255).astype(np.uint8)
        out = AUG_STATE["aug"](image=np_img)["image"]
        out_t = torch.from_numpy(out).to(im.device).float().permute(2,0,1)/255.0
        imgs[i] = out_t; applied += 1

    if scaled_to_01: imgs *= 255.0
    batch["img"] = imgs

# ==================================================



def val_with_custom_val(model: YOLO, base_yaml: str, custom_val_source: str, save_subname: str,
                        imgsz: int, batch: int, save_dir: str):
    """
    用“替换 val 字段”的临时 yaml 做分域验证（day/night 列表）。
    """
    with open(base_yaml, "r", encoding="utf-8") as f:
        data = yaml.safe_load(f)
    data["val"] = custom_val_source
    tmp_yaml = os.path.join(save_dir, f"_{save_subname}.yaml")
    with open(tmp_yaml, "w", encoding="utf-8") as f:
        yaml.safe_dump(data, f, allow_unicode=True, sort_keys=False)
    metrics = model.val(
        data=tmp_yaml, imgsz=imgsz, batch=batch, device=cfg["device"],
        project=save_dir, name=save_subname, plots=True, save_json=True
    )
    return metrics



def metrics_to_dict(m):
    try:
        return {
            "map50-95": float(m.box.map),
            "map50": float(m.box.map50),
            "map75": float(m.box.map75),
            "maps_per_class": [float(x) for x in m.box.maps] if getattr(m.box, "maps", None) is not None else None,
            "precision": float(getattr(m.box, "mp", np.mean(m.box.p))),
            "recall": float(getattr(m.box, "mr", np.mean(m.box.r))),
        }
    except Exception as e:
        return m if isinstance(m, dict) else {"error": type(e).__name__, "message": str(e)}


if __name__ == "__main__":
    # ====== Train ======
    model = YOLO(cfg["model"])
    model.add_callback("on_train_epoch_start", on_train_epoch_start)
    model.add_callback("on_train_batch_start", on_train_batch_start)
    train_results = model.train(**cfg)

    # 训练输出目录
    save_dir = str(model.trainer.save_dir)

    # 保存训练配置
    with open(os.path.join(save_dir, "train_config.json"), "w", encoding="utf-8") as f:
        json.dump(cfg, f, indent=2)

    # ====== Val (overall val split) ======
    metrics_val = model.val(
        data=DATA_BASE_YAML, split="val",
        imgsz=cfg["imgsz"], batch=cfg["batch"], device=cfg["device"],
        project=save_dir, name="val_all", plots=True, save_json=True
    )

    # ====== Val (day / night separately, 若清单存在) ======
    metrics_day = metrics_night = None
    if os.path.isfile(VAL_DAY_LIST):
        metrics_day = val_with_custom_val(model, DATA_BASE_YAML, VAL_DAY_LIST, "val_day",
                                          cfg["imgsz"], cfg["batch"], save_dir)
    if os.path.isfile(VAL_NIGHT_LIST):
        metrics_night = val_with_custom_val(model, DATA_BASE_YAML, VAL_NIGHT_LIST, "val_night",
                                            cfg["imgsz"], cfg["batch"], save_dir)

    # ====== Test (official test split) ======
    metrics_test = model.val(
        data=DATA_BASE_YAML, split="test",
        imgsz=cfg["imgsz"], batch=cfg["batch"], device=cfg["device"],
        project=save_dir, name="test", plots=True, save_json=True
    )

    # ====== Save metrics summary ======
    summary = {
        "val_all":  metrics_to_dict(metrics_val),
        "val_day":  metrics_to_dict(metrics_day) if metrics_day else None,
        "val_night":metrics_to_dict(metrics_night) if metrics_night else None,
        "test":     metrics_to_dict(metrics_test)
    }
    with open(os.path.join(save_dir, "metrics_summary.json"), "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)
    print("Saved metrics to", os.path.join(save_dir, "metrics_summary.json"))


Ultralytics 8.3.179 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/mixed/mixed/daynight.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.15, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.5, hsv_v=0.2, imgsz=896, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=0.7, multi_scale=False, name=dn_baseline, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=30, perspective=0.0, plots=True, pose=12.0, pre

train: Scanning /content/mixed/mixed/labels/train... 1172 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1172/1172 [00:00<00:00, 1543.19it/s]

train: New cache created: /content/mixed/mixed/labels/train.cache
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 535.4±636.2 MB/s, size: 420.3 KB)


val: Scanning /content/mixed/mixed/labels/val... 146 images, 0 backgrounds, 0 corrupt: 100%|██████████| 146/146 [00:00<00:00, 410.65it/s]

...

[NightAug] epoch=43 scale=0.00

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      43/50      6.84G     0.7552     0.4723      1.081         19        896: 100%|██████████| 74/74 [00:45<00:00,  1.63it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 5/5 [00:04<00:00,  1.01it/s]

                   all        146        427      0.676      0.528       0.61      0.408


[NightAug] epoch=44 scale=0.00

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      44/50      6.84G     0.7216     0.4581      1.065         41        896:  77%|███████▋  | 57/74 [00:36<00:09,  1.83it/s]

In [17]:
p = os.path.join("runs_daynight", sorted(os.listdir("runs_daynight"))[-1], "metrics_summary.json")
m = json.load(open(p, "r", encoding="utf-8"))
def val(x): return None if x is None else float(x["map50-95"])
print("mAP50-95  val_all =", val(m["val_all"]))
print("mAP50-95  val_day =", val(m["val_day"]))
print("mAP50-95  val_night =", val(m["val_night"]))
if m["val_day"] and m["val_night"]:
    print("ΔmAP =", abs(val(m["val_day"]) - val(m["val_night"])))


mAP50-95  val_all = 0.3737483860002898
mAP50-95  val_day = 0.9077373729181109
mAP50-95  val_night = 0.3134976150478496
ΔmAP = 0.5942397578702612


## origin

In [ ]:
from ultralytics import YOLO
import torch
import json
from datetime import datetime
import os

# ==== settings ====
now_str = datetime.now().strftime('%Y%m%d_%H%M%S')
'''config = {
    "model": "yolov8n.pt",
    "data": "/content/mixed_ct/mixed_ct/data.yaml",
    "epochs": 50,# 50
    "imgsz": 768,# 320
    "batch": 32,
    "name": f"furniture_yolov8n_{now_str}",
    "project": "furniture_project_3",
    "exist_ok": True,
    "device": "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu",
    "lr0": 0.01,
    "lrf": 0.01,
    "momentum": 0.937,
    "weight_decay": 0.0005,
    "warmup_epochs": 3.0,
    "warmup_momentum": 0.8,
    "warmup_bias_lr": 0.1,
    "box": 7.5,
    "cls": 0.5,
    "dfl": 1.5,
    "degrees": 0.0,
    "translate": 0.1,
    "scale": 0.5,
    "shear": 0.0,
    "perspective": 0.0,
    "flipud": 0.0,
    "fliplr": 0.5,
    "mosaic": 1.0,
    "mixup": 0.05, # 0
    "copy_paste": 0.0,
    "patience": 50,
    "workers": 8,
    "save": True,
    "save_period": -1,
    "cache": False,
    "close_mosaic": 10,# 0
    "resume": False,
    "amp": True,
    "pretrained": True,
    "erasing": 0.15, #
    "workers" : 8 #


}'''
config = {
    "model": "yolov8n.pt",
    "data": "/content/mixed_ct/mixed_ct/data.yaml",
    "epochs": 50,
    "imgsz": 640,
    "batch": 16,
    "lr0": 0.01,
    "lrf": 0.01,
    "optimizer": "SGD",      # 关键：换回 SGD
    "momentum": 0.937,
    "weight_decay": 0.0005,
    "cos_lr": False,
    "warmup_epochs": 3.0,
    "warmup_momentum": 0.8,
    "warmup_bias_lr": 0.1,

    "mosaic": 1.0,
    "close_mosaic": 10,
    "mixup": 0.05,
    "copy_paste": 0.2,
    "erasing": 0.15,
    "fliplr": 0.5,
    "flipud": 0.0,
    "degrees": 0.0, "translate": 0.1, "scale": 0.5, "shear": 0.0, "perspective": 0.0,

    "hsv_h": 0.015, "hsv_s": 0.7, "hsv_v": 0.4,

    "rect": False,
    "patience": 50,
    "workers": 8,
    "amp": True,
    "pretrained": True,
    "project": "furniture_project_3",
    "exist_ok": True,
}

# ==== Train ====
model = YOLO(config["model"])
results = model.train(**config)

# ==== Save ====
save_dir = model.trainer.save_dir
config_path = os.path.join(save_dir, "config.json")
with open(config_path, "w") as f:
    json.dump(config, f, indent=4)
print(f"param saved: {config_path}")

New https://pypi.org/project/ultralytics/8.3.178 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.177 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.2, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/mixed_ct/mixed_ct/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.15, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.05, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimiz

train: Scanning /content/mixed_ct/mixed_ct/train/labels.cache... 1108 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1108/1108 [00:00<?, ?it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 260.7±139.1 MB/s, size: 84.0 KB)


val: Scanning /content/mixed_ct/mixed_ct/val/labels.cache... 177 images, 0 backgrounds, 0 corrupt: 100%|██████████| 177/177 [00:00<?, ?it/s]


...

      50/50      3.04G     0.9865     0.8589      1.168          6        640: 100%|██████████| 70/70 [00:26<00:00,  2.66it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 6/6 [00:01<00:00,  3.38it/s]

                   all        177        477      0.608      0.524      0.573      0.394



50 epochs completed in 0.409 hours.
Optimizer stripped from furniture_project_3/train/weights/last.pt, 6.2MB
Optimizer stripped from furniture_project_3/train/weights/best.pt, 6.2MB

Validating furniture_project_3/train/weights/best.pt...
Ultralytics 8.3.177 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 72 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 6/6 [00:04<00:00,  1.49it/s]


                   all        177        477       0.65      0.515      0.582      0.401
                 Chair        114        228      0.714      0.583       0.66      0.495
                 Table         63        249      0.585      0.446      0.504      0.307
Speed: 0.3ms preprocess, 2.5ms inference, 0.0ms loss, 7.0ms postprocess per image
Results saved to furniture_project_3/train
param saved: furniture_project_3/train/config.json
